# Hybrid Recommendation with Product Title Embeddings

This notebook extends the metadata-based hybrid recommender by adding SBERT embeddings of **product titles** from the metadata.

**Experiment Plan:**
1. Load existing data splits and mappings
2. Compute SBERT embeddings for **product titles** (from metadata only - not review titles)
3. Add lightweight projection layer to map SBERT embeddings → 64D model space
4. Train hybrid model with text embeddings for 1-3 epochs
5. Evaluate HR@10, NDCG@10 vs metadata-only baseline
6. Analyze cold-item performance (items with ≤5 ratings)

**Note:** We use product titles from metadata (actual product names) rather than review titles (user-written review headings).

### Additive Metadata Embeddings
Use item-user interaction with Review signals, category, and price along with text embeddings 

In [8]:
import pandas as pd
import numpy as np

# Load the final filtered dataset
review_data = pd.read_json('../data/processed/review_data.jsonl', lines=True)
metadata = pd.read_json('../data/processed/metadata.jsonl', lines=True)

print(f"Loaded {len(review_data)} reviews and {len(metadata)} metadata records")
print("Review data columns:", review_data.columns.tolist())
print("Metadata columns:", metadata.columns.tolist())

# Check product title availability in metadata (this is what we'll use for embeddings)
print(f"\nProduct title availability in metadata:")
print(f"Product titles: {metadata['title'].notna().sum()}/{len(metadata)} ({100*metadata['title'].notna().mean():.1f}%)")

# Sample product titles to understand the data
print(f"\nSample product titles from metadata:")
sample_titles = metadata['title'].dropna().head(5).tolist()
for i, title in enumerate(sample_titles, 1):
    print(f"{i}. {title}")

# Note: We'll ignore review titles as they are user-written review titles, not product names

Loaded 390757 reviews and 136934 metadata records
Review data columns: ['user_id', 'parent_asin', 'rating', 'timestamp', 'verified_purchase', 'helpful_vote', 'text', 'title', 'reviewTime', 'days_since_start']
Metadata columns: ['main_category', 'title', 'average_rating', 'rating_number', 'features', 'description', 'price', 'images', 'videos', 'categories', 'details', 'parent_asin', 'bought_together']

Product title availability in metadata:
Product titles: 136934/136934 (100.0%)

Sample product titles from metadata:
1. NotoCity Compatible with Vivoactive 4 band 22mm Quick Release Silicone Bands/Garmin Darth Vader/First Avenger/Polar Vantage Smartwatch Sport Breathable Strap Replacement for Gear S3 Classic Watchband
2. Motorola Droid X Essentials Combo Pack
3. QGHXO Band for Garmin Vivofit 4, Soft Silicone Replacement Watch Band Strap for Garmin Vivofit 4 Activity Tracker, Small, Large, Ten Colors (5PCS Bands-Girl, Large)
4. KEiiD PC Computer Speaker Compact Bluetooth Stereo System with

In [9]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
import torch

# Assume df has ['reviewerID', 'asin', 'overall', 'reviewTime']
ratings_df = review_data[['user_id', 'parent_asin', 'rating', 'reviewTime']].copy()
# Include title in metadata_df for SBERT embeddings
metadata_df = metadata[['parent_asin', 'main_category', 'average_rating', 'rating_number', 'price', 'title']].copy()

print("Ratings data shape:", ratings_df.shape)
print("Metadata data shape:", metadata_df.shape)

# Step 1 - Data Preparation

# Map users/items to integer IDs.
user2idx = {u: i for i, u in enumerate(ratings_df['user_id'].unique())}
item2idx = {i: j for j, i in enumerate(ratings_df['parent_asin'].unique())}

# Add the encoded columns
ratings_df['user_idx'] = ratings_df['user_id'].map(user2idx)
ratings_df['item_idx'] = ratings_df['parent_asin'].map(item2idx)

users = torch.tensor(ratings_df['user_idx'].values)
items = torch.tensor(ratings_df['item_idx'].values)
ratings = torch.tensor(ratings_df['rating'].values, dtype=torch.float32)

## Metadata Preparation
metadata_df['price'] = pd.to_numeric(metadata_df['price'], errors='coerce')
metadata_df['average_rating'] = pd.to_numeric(metadata_df['average_rating'], errors='coerce').fillna(0)
metadata_df['rating_number'] = pd.to_numeric(metadata_df['rating_number'], errors='coerce').fillna(0).astype(int)

# Transform numeric fields (log + scale where it makes sense)
metadata_df['price_log'] = np.log1p(metadata_df['price'])
metadata_df['rating_number_log'] = np.log1p(metadata_df['rating_number'])

scalers = {}
for col in ['price_log', 'average_rating', 'rating_number_log']:
    scaler = StandardScaler()
    metadata_df[col + '_scaled'] = scaler.fit_transform(metadata_df[[col]])
    scalers[col] = scaler

# Map main_category to integer IDs
main_categories = metadata_df['main_category'].fillna('Unknown').astype(str)
cat2idx = {cat: idx+1 for idx, cat in enumerate(main_categories.unique())}
cat2idx['<unk>'] = 0
metadata_df['main_cat_idx'] = main_categories.map(lambda c: cat2idx.get(c, 0))

# Merge on parent_asin
merged_df = ratings_df.merge(
    metadata_df[['parent_asin', 'main_cat_idx', 
                 'price_log_scaled', 'average_rating_scaled', 'rating_number_log_scaled']],
    on='parent_asin',
    how='left'
)

print("Merged data shape:", merged_df.shape)
print("Sample merged data:")
print(merged_df.head())

Ratings data shape: (390757, 4)
Metadata data shape: (136934, 6)
Merged data shape: (390757, 10)
Sample merged data:
                        user_id parent_asin  rating    reviewTime  user_idx  \
0  AG4ZJVVMOHDHXHN3WXQWNZU3NNPQ  B00000JBAT     2.0  931886331000         0   
1  AENJWAP4JGEFZGDCTSX72UQ6M7IQ  B00000JDHV     4.0  940168313000         1   
2  AEPLJC56FLQDFUA7ONJMXQVCWWUQ  B00000SG9M     4.0  940962345000         2   
3  AGJR3BPX2WZDSXGUEX5CCKXKPOFA  B00000J4FY     4.0  945781000000         3   
4  AE2Z5HN5SLQ73KR34QJFWLH2BREA  B00002JXBI     5.0  947544621000         4   

   item_idx  main_cat_idx  price_log_scaled  average_rating_scaled  \
0         0             5         -0.165522              -2.670381   
1         1             3         -0.165522              -1.746734   
2         2             5         -0.165522              -2.670381   
3         3             5         -0.165522              -0.592176   
4         4             3          1.160006              -

In [10]:
print(merged_df.head())
print("Users:", merged_df['user_id'].nunique())
print("Items:", merged_df['parent_asin'].nunique())
print("Categories:", len(cat2idx))

                        user_id parent_asin  rating    reviewTime  user_idx  \
0  AG4ZJVVMOHDHXHN3WXQWNZU3NNPQ  B00000JBAT     2.0  931886331000         0   
1  AENJWAP4JGEFZGDCTSX72UQ6M7IQ  B00000JDHV     4.0  940168313000         1   
2  AEPLJC56FLQDFUA7ONJMXQVCWWUQ  B00000SG9M     4.0  940962345000         2   
3  AGJR3BPX2WZDSXGUEX5CCKXKPOFA  B00000J4FY     4.0  945781000000         3   
4  AE2Z5HN5SLQ73KR34QJFWLH2BREA  B00002JXBI     5.0  947544621000         4   

   item_idx  main_cat_idx  price_log_scaled  average_rating_scaled  \
0         0             5         -0.165522              -2.670381   
1         1             3         -0.165522              -1.746734   
2         2             5         -0.165522              -2.670381   
3         3             5         -0.165522              -0.592176   
4         4             3          1.160006              -2.670381   

   rating_number_log_scaled  
0                 -0.509908  
1                 -0.944785  
2             

In [11]:
# Train test split
# For each user, keep their last review as test, and earlier ones as train.
#sort by time
merged_df = merged_df.sort_values(by=['user_id', 'reviewTime'])

#Split
test_df = merged_df.groupby('user_id').tail(1)
train_df = merged_df.drop(test_df.index)


In [12]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AdditiveHybridMFWithText(nn.Module):
    def __init__(self,
                 n_users,
                 n_items,
                 n_main_cats,
                 text_emb_dim,
                 emb_dim=64,
                 use_avg_rating=True,
                 use_rating_count=True,
                 use_price=True,
                 use_text=True,
                 dropout=0.0):
        super().__init__()
        self.emb_dim = emb_dim
        self.use_text = use_text

        # main embeddings
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.item_emb = nn.Embedding(n_items, emb_dim)
        self.cat_emb  = nn.Embedding(n_main_cats, emb_dim)   # 0 reserved for <unk>

        # numeric feature projections (scalar -> embedding)
        self.use_price = use_price
        self.use_avg_rating = use_avg_rating
        self.use_rating_count = use_rating_count

        if self.use_price:
            self.price_proj = nn.Linear(1, emb_dim, bias=True)
        else:
            self.price_proj = None

        if self.use_avg_rating:
            self.avg_proj = nn.Linear(1, emb_dim, bias=True)
        else:
            self.avg_proj = None

        if self.use_rating_count:
            self.count_proj = nn.Linear(1, emb_dim, bias=True)
        else:
            self.count_proj = None

        # text projection (SBERT dim -> embedding dim)
        if self.use_text:
            self.text_proj = nn.Linear(text_emb_dim, emb_dim, bias=True)
        else:
            self.text_proj = None

        # biases
        self.user_bias = nn.Embedding(n_users, 1)
        self.item_bias = nn.Embedding(n_items, 1)

        # optional dropout on item vector
        self.dropout = nn.Dropout(dropout) if dropout > 0 else None

        # initialization (small normal)
        self._init_weights()

    def _init_weights(self):
        std = 0.01
        for emb in (self.user_emb, self.item_emb, self.cat_emb):
            nn.init.normal_(emb.weight, mean=0.0, std=std)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.item_bias.weight)
        # Linear proj init
        for proj in [self.price_proj, self.avg_proj, self.count_proj, self.text_proj]:
            if proj is not None:
                nn.init.xavier_uniform_(proj.weight)
                nn.init.zeros_(proj.bias)

    def forward_item_vector(self, item_idx, main_cat_idx, price_val=None,
                            avg_rating_val=None, rating_count_val=None, text_emb=None):
        v_item = self.item_emb(item_idx)           # (B, D)
        v_cat  = self.cat_emb(main_cat_idx)        # (B, D)
        parts = [v_item, v_cat]

        if self.price_proj is not None and price_val is not None:
            # ensure shape (B,1)
            pv = price_val.view(-1, 1).float()
            v_price = self.price_proj(pv)         # (B, D)
            parts.append(v_price)

        if self.avg_proj is not None and avg_rating_val is not None:
            av = avg_rating_val.view(-1, 1).float()
            v_avg = self.avg_proj(av)
            parts.append(v_avg)

        if self.count_proj is not None and rating_count_val is not None:
            cv = rating_count_val.view(-1, 1).float()
            v_count = self.count_proj(cv)
            parts.append(v_count)

        if self.text_proj is not None and text_emb is not None:
            v_text = self.text_proj(text_emb)     # (B, D)
            parts.append(v_text)

        item_vec = sum(parts)  # additive combination

        if self.dropout is not None:
            item_vec = self.dropout(item_vec)

        return item_vec

    def score(self, user_idx, item_idx, main_cat_idx, price_val=None,
              avg_rating_val=None, rating_count_val=None, text_emb=None):

        u = self.user_emb(user_idx)                 # (B, D)
        item_vec = self.forward_item_vector(item_idx, main_cat_idx,
                                            price_val, avg_rating_val, rating_count_val, text_emb)  # (B, D)
        dot = (u * item_vec).sum(dim=-1)            # (B,)
        b_u = self.user_bias(user_idx).squeeze(-1)  # (B,)
        b_i = self.item_bias(item_idx).squeeze(-1)  # (B,)
        return dot + b_u + b_i

    def forward(self, user_idx, item_idx, main_cat_idx, price_val=None,
                avg_rating_val=None, rating_count_val=None, text_emb=None):

        return self.score(user_idx, item_idx, main_cat_idx, price_val, avg_rating_val, rating_count_val, text_emb)

In [13]:
import torch
import torch.nn.functional as F
import torch.optim as optim
import torch.nn as nn
import numpy as np
from time import time

# Setup device first
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

# --- Embedding size bases (use max index + 1 to cover full index space incl 0) --- #
n_users = int(train_df['user_idx'].max()) + 1
n_items = int(train_df['item_idx'].max()) + 1
n_main_cats = int(train_df['main_cat_idx'].max()) + 1  # includes 0 = <unk>

# Create idx2item mapping for title lookup
idx2item = {j: i for i, j in item2idx.items()}

# Also check if test set has indices beyond training range
test_max_users = int(test_df['user_idx'].max()) + 1
test_max_items = int(test_df['item_idx'].max()) + 1
test_max_cats = int(test_df['main_cat_idx'].max()) + 1

if test_max_users > n_users or test_max_items > n_items or test_max_cats > n_main_cats:
    print(f"Train ranges: users={n_users}, items={n_items}, cats={n_main_cats}")
    print(f"Test ranges: users={test_max_users}, items={test_max_items}, cats={test_max_cats}")
    # Use max from both train and test to be safe
    n_users = max(n_users, test_max_users)
    n_items = max(n_items, test_max_items)
    n_main_cats = max(n_main_cats, test_max_cats)

print(f"Final embedding sizes: n_users={n_users} n_items={n_items} n_main_cats={n_main_cats}")

# Set SBERT dimension (will be updated after SBERT model loads)
sbert_dim = 384  # Default for all-MiniLM-L6-v2, will be updated in SBERT cell


# 3. Text-enhanced model
text_model = AdditiveHybridMFWithText(
    n_users=n_users,
    n_items=n_items,
    n_main_cats=n_main_cats,
    text_emb_dim=sbert_dim,
    emb_dim=64,
    use_avg_rating=True,
    use_rating_count=True,
    use_price=True,
    use_text=True,  # Include text embeddings
    dropout=0.1
)

text_model.to(device)

# --- Build item-level metadata arrays (use ONLY train data to avoid leakage) --- #
item_cat = np.zeros(n_items, dtype=np.int64)
item_price = np.zeros(n_items, dtype=np.float32)
item_avg = np.zeros(n_items, dtype=np.float32)
item_count = np.zeros(n_items, dtype=np.float32)

meta_source = (
    train_df[['item_idx','main_cat_idx','price_log_scaled','average_rating_scaled','rating_number_log_scaled']]
    .drop_duplicates('item_idx')
)
for _, row in meta_source.iterrows():
    i = int(row.item_idx)
    if i < n_items:  # safety
        item_cat[i] = int(row.main_cat_idx) if pd.notna(row.main_cat_idx) else 0
        item_price[i] = float(row.price_log_scaled) if pd.notna(row.price_log_scaled) else 0.0
        item_avg[i] = float(row.average_rating_scaled) if pd.notna(row.average_rating_scaled) else 0.0
        item_count[i] = float(row.rating_number_log_scaled) if pd.notna(row.rating_number_log_scaled) else 0.0

# Check for any remaining NaN values and fill with zeros
item_price = np.nan_to_num(item_price, nan=0.0)
item_avg = np.nan_to_num(item_avg, nan=0.0) 
item_count = np.nan_to_num(item_count, nan=0.0)

# Convert to tensors & move to device once
item_cat_t = torch.from_numpy(item_cat).to(device)
item_price_t = torch.from_numpy(item_price).to(device)
item_avg_t = torch.from_numpy(item_avg).to(device)
item_count_t = torch.from_numpy(item_count).to(device)

# Training interaction tensors (kept on CPU until indexing then moved in batch)
u_arr = torch.from_numpy(train_df['user_idx'].values.astype(np.int64))
pos_item_arr = torch.from_numpy(train_df['item_idx'].values.astype(np.int64))
pos_cat_arr = torch.from_numpy(train_df['main_cat_idx'].values.astype(np.int64))
pos_price_arr = torch.from_numpy(train_df['price_log_scaled'].values.astype(np.float32))
pos_avg_arr   = torch.from_numpy(train_df['average_rating_scaled'].values.astype(np.float32))
pos_count_arr = torch.from_numpy(train_df['rating_number_log_scaled'].values.astype(np.float32))

n_train = u_arr.shape[0]


# Replace any NaN values with 0
pos_price_arr = torch.nan_to_num(pos_price_arr, nan=0.0)
pos_avg_arr = torch.nan_to_num(pos_avg_arr, nan=0.0)
pos_count_arr = torch.nan_to_num(pos_count_arr, nan=0.0)

# Build user->set(positive items) for better negative sampling (train only)
user_pos = {}
for u,i in zip(u_arr.numpy(), pos_item_arr.numpy()):
    user_pos.setdefault(int(u), set()).add(int(i))



Device: cuda
Train ranges: users=41909, items=136930, cats=35
Test ranges: users=41909, items=136934, cats=33
Final embedding sizes: n_users=41909 n_items=136934 n_main_cats=35


In [14]:
# Step 2: Compute SBERT Product Title Embeddings
import os
from sentence_transformers import SentenceTransformer

# Load SBERT model (all-MiniLM-L6-v2 is fast and good for general text)
print("Loading SBERT model...")
sbert_model = SentenceTransformer('all-MiniLM-L6-v2')
sbert_dim = sbert_model.get_sentence_embedding_dimension()
print(f"SBERT embedding dimension: {sbert_dim}")

# Create embeddings directory
os.makedirs('../data/embeddings', exist_ok=True)

# Prepare product titles from metadata only
# Fill missing titles with 'Unknown Product'
metadata_titles = metadata_df[['parent_asin', 'title']].copy()
metadata_titles['title'] = metadata_titles['title'].fillna('Unknown Product').astype(str)


# Build title array aligned with item_idx (using only product titles from metadata)
item_titles = ['Unknown Product'] * n_items
title_lookup = dict(zip(metadata_titles['parent_asin'], metadata_titles['title']))

for idx in range(n_items):
    if idx in idx2item:
        asin = idx2item[idx]
        item_titles[idx] = title_lookup.get(asin, 'Unknown Product')


# Compute embeddings (this may take a few minutes)
print("\nComputing SBERT embeddings for product titles...")
title_embeddings = sbert_model.encode(item_titles, show_progress_bar=True, convert_to_numpy=True)
print(f"Computed embeddings shape: {title_embeddings.shape}")

# Save embeddings 
np.save('../data/embeddings/title_embeddings_hybrid.npy', title_embeddings)
print("Saved embeddings to ../data/embeddings/title_embeddings_hybrid.npy")

# Convert to torch tensor and move to device
title_embeddings_tensor = torch.from_numpy(title_embeddings).float().to(device)
print(f"Title embeddings on device: {title_embeddings_tensor.shape}")

Loading SBERT model...
SBERT embedding dimension: 384

Computing SBERT embeddings for product titles...


Batches: 100%|██████████| 4280/4280 [01:21<00:00, 52.69it/s]


Computed embeddings shape: (136934, 384)
Saved embeddings to ../data/embeddings/title_embeddings_hybrid.npy
Title embeddings on device: torch.Size([136934, 384])


In [15]:
# Training with BPR Loss
# BPR loss
softplus = nn.Softplus()

def bpr_loss(pos_scores, neg_scores):
    return softplus(neg_scores - pos_scores).mean()

def sample_neg(user_batch, n_items, user_pos_sets, device):
    """Sample one negative per user avoiding known positives (simple rejection)."""
    B = user_batch.size(0)
    neg = torch.randint(0, n_items, (B,), device=device)
    for idx, u in enumerate(user_batch.tolist()):
        tries = 0
        while neg[idx].item() in user_pos_sets.get(u, ()) and tries < 10:
            neg[idx] = torch.randint(0, n_items, (1,), device=device)
            tries += 1
    return neg

def train_bpr_with_text(model, epochs=3, batch_size=2048, l2_lambda=0.0, model_type="CF"):
    """Train with different model types: CF, Baseline, or Text"""
    n = n_train
    indices = np.arange(n)
    model.train()
    
    # Optimizer
    decay, no_decay = [], []
    for name, p in model.named_parameters():
        if not p.requires_grad: 
            continue
        if p.ndim == 1 or name.endswith(".bias"):
            no_decay.append(p)
        else:
            decay.append(p)
    optimizer = optim.AdamW([{'params': decay, 'weight_decay': 1e-5},
                             {'params': no_decay, 'weight_decay': 0.0}], lr=5e-4)
    
    for epoch in range(1, epochs+1):
        np.random.shuffle(indices)
        epoch_loss = 0.0
        seen = 0
        t0 = time()
        for start in range(0, n, batch_size):
            batch_idx = indices[start:start+batch_size]
            if len(batch_idx) == 0:
                continue
            user = u_arr[batch_idx].to(device)
            pos_item = pos_item_arr[batch_idx].to(device)
            
            neg_item = sample_neg(user, n_items, user_pos, device)
            optimizer.zero_grad()
            
            if model_type == "CF":
                # Basic CF model - only user and item embeddings
                pos_scores = model.score(user, pos_item)
                neg_scores = model.score(user, neg_item)
            elif model_type == "Text":
                # Text-enhanced model with all features
                pos_cat  = pos_cat_arr[batch_idx].to(device)
                pos_price = pos_price_arr[batch_idx].to(device)
                pos_avg   = pos_avg_arr[batch_idx].to(device)
                pos_count = pos_count_arr[batch_idx].to(device)
                
                neg_cat = item_cat_t[neg_item]
                neg_price = item_price_t[neg_item]
                neg_avg = item_avg_t[neg_item]
                neg_count = item_count_t[neg_item]
                
                # Get text embeddings for positive and negative items
                pos_text = title_embeddings_tensor[pos_item]
                neg_text = title_embeddings_tensor[neg_item]
                pos_scores = model.score(user, pos_item, pos_cat, pos_price, pos_avg, pos_count, pos_text)
                neg_scores = model.score(user, neg_item, neg_cat, neg_price, neg_avg, neg_count, neg_text)
            else:
                # Baseline model with metadata but no text
                pos_cat  = pos_cat_arr[batch_idx].to(device)
                pos_price = pos_price_arr[batch_idx].to(device)
                pos_avg   = pos_avg_arr[batch_idx].to(device)
                pos_count = pos_count_arr[batch_idx].to(device)
                
                neg_cat = item_cat_t[neg_item]
                neg_price = item_price_t[neg_item]
                neg_avg = item_avg_t[neg_item]
                neg_count = item_count_t[neg_item]
                
                pos_scores = model.score(user, pos_item, pos_cat, pos_price, pos_avg, pos_count)
                neg_scores = model.score(user, neg_item, neg_cat, neg_price, neg_avg, neg_count)
                
            loss = bpr_loss(pos_scores, neg_scores)
            if l2_lambda > 0:
                # light embedding norm penalty
                l2 = (model.user_emb(user).pow(2).mean() +
                      model.item_emb(pos_item).pow(2).mean())
                if hasattr(model, 'item_emb'):  # For CF model
                    l2 += model.item_emb(neg_item).pow(2).mean()
                loss = loss + l2_lambda * l2
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * len(batch_idx)
            seen += len(batch_idx)
        avg = epoch_loss / max(1, seen)
        print(f"{model_type} Epoch {epoch}/{epochs}  BPR_Loss={avg:.5f}  time={time()-t0:.1f}s")
    print(f"{model_type} training complete.")

# Load title embeddings if not already available
try:
    title_embeddings_tensor
    print("Title embeddings already loaded in memory")
except NameError:
    print("Loading title embeddings from saved file...")
    title_embeddings = np.load('../data/embeddings/title_embeddings.npy')
    title_embeddings_tensor = torch.from_numpy(title_embeddings).float().to(device)
    print(f"Loaded title embeddings: {title_embeddings_tensor.shape}")

print("\n=== Training Text Model (Metadata + Text Embeddings) ===")
train_bpr_with_text(text_model, epochs=10, batch_size=2048, l2_lambda=1e-6, model_type="Text")


Title embeddings already loaded in memory

=== Training Text Model (Metadata + Text Embeddings) ===
Text Epoch 1/10  BPR_Loss=0.66287  time=15.5s
Text Epoch 2/10  BPR_Loss=0.55605  time=11.5s
Text Epoch 3/10  BPR_Loss=0.48056  time=10.4s
Text Epoch 4/10  BPR_Loss=0.43528  time=10.3s
Text Epoch 5/10  BPR_Loss=0.40026  time=10.8s
Text Epoch 6/10  BPR_Loss=0.37006  time=11.0s
Text Epoch 7/10  BPR_Loss=0.34364  time=9.8s
Text Epoch 8/10  BPR_Loss=0.32249  time=10.4s
Text Epoch 9/10  BPR_Loss=0.30453  time=10.9s
Text Epoch 10/10  BPR_Loss=0.29080  time=12.8s
Text training complete.


In [17]:
# Comprehensive Bucketed Evaluation for the model on the full item catalog.
import math

def bucketed_evaluation(models_dict, test_df, K=10):

    # Calculate item popularity from training data
    item_popularity = train_df['item_idx'].value_counts().to_dict()
    
    # Define buckets based on training popularity
    cold_items = set([item for item, count in item_popularity.items() if count <= 5])
    warm_items = set([item for item, count in item_popularity.items() if 6 <= count <= 50])
    hot_items = set([item for item, count in item_popularity.items() if count > 50])
    
    print(f"Item Popularity Distribution:")
    print(f"Cold items (≤5 ratings): {len(cold_items)}")
    print(f"Warm items (6-50 ratings): {len(warm_items)}")
    print(f"Hot items (>50 ratings): {len(hot_items)}")
    print(f"Total items: {len(item_popularity)}")
    
    # Build user training items mapping for filtering
    user_train_items = {}
    for _, row in train_df.iterrows():
        u = int(row['user_idx'])
        item = int(row['item_idx'])
        if u not in user_train_items:
            user_train_items[u] = set()
        user_train_items[u].add(item)
    
    # Filter test set to users who appear in training
    train_users_set = set(train_df['user_idx'].unique())
    test_df_filtered = test_df[test_df['user_idx'].isin(train_users_set)].copy()
    
    print(f"\nTest users after filtering: {len(test_df_filtered)}")
    
    # Categorize test items into buckets
    test_df_filtered['item_bucket'] = test_df_filtered['item_idx'].apply(
        lambda x: 'cold' if x in cold_items else 'warm' if x in warm_items else 'hot'
    )
    
    bucket_counts = test_df_filtered['item_bucket'].value_counts()
    print(f"\nTest item distribution:")
    for bucket in ['cold', 'warm', 'hot']:
        count = bucket_counts.get(bucket, 0)
        print(f"{bucket.capitalize()}: {count} test interactions")
    
    # Results storage
    results = {}
    
    # Evaluate each model
    for model_name, model in models_dict.items():
        print(f"\n{'='*60}")
        print(f"EVALUATING {model_name.upper()}")
        print(f"{'='*60}")
        
        model.eval()
        all_items_tensor = torch.arange(n_items, device=device)
        
        # Initialize metrics per bucket
        bucket_metrics = {
            'cold': {'hits': [], 'ndcgs': []},
            'warm': {'hits': [], 'ndcgs': []},
            'hot': {'hits': [], 'ndcgs': []},
            'overall': {'hits': [], 'ndcgs': []}
        }
        
        with torch.no_grad():
            for _, row in test_df_filtered.iterrows():
                u = int(row['user_idx'])
                true_item = int(row['item_idx'])
                bucket = row['item_bucket']
                
                if u >= n_users or true_item >= n_items:
                    continue
                
                # Score all items for this user
                user_tensor = torch.full((n_items,), u, device=device, dtype=torch.long)
                
                # Get scores based on model type
                if model_name == "Basic CF":
                    scores = model.score(user_tensor, all_items_tensor)
                else:  # Hybrid models
                    if model_name == "Text Model":
                        # Use text embeddings
                        scores = model.score(user_tensor, all_items_tensor, item_cat_t, 
                                           item_price_t, item_avg_t, item_count_t, 
                                           title_embeddings_tensor)
                    else:  # Baseline Model
                        # No text embeddings
                        scores = model.score(user_tensor, all_items_tensor, item_cat_t, 
                                           item_price_t, item_avg_t, item_count_t)
                
                # Mask training items by setting their scores to -inf
                train_items = user_train_items.get(u, set())
                for train_item in train_items:
                    if train_item < len(scores):
                        scores[train_item] = float('-inf')
                
                # Get top-K recommendations
                _, top_indices = torch.topk(scores, k=K, largest=True)
                top_items = top_indices.cpu().numpy().tolist()
                
                # Calculate metrics
                hit = 1.0 if true_item in top_items else 0.0
                
                # NDCG calculation
                if true_item in top_items:
                    rank = top_items.index(true_item) + 1
                    ndcg = 1.0 / math.log2(rank + 1)
                else:
                    ndcg = 0.0
                
                # Store metrics for bucket and overall
                bucket_metrics[bucket]['hits'].append(hit)
                bucket_metrics[bucket]['ndcgs'].append(ndcg)
                bucket_metrics['overall']['hits'].append(hit)
                bucket_metrics['overall']['ndcgs'].append(ndcg)
        
        # Calculate final metrics for this model
        model_results = {}
        for bucket in ['cold', 'warm', 'hot', 'overall']:
            if len(bucket_metrics[bucket]['hits']) > 0:
                hr = np.mean(bucket_metrics[bucket]['hits'])
                ndcg = np.mean(bucket_metrics[bucket]['ndcgs'])
                count = len(bucket_metrics[bucket]['hits'])
            else:
                hr = ndcg = count = 0
            
            model_results[bucket] = {
                'hr': hr,
                'ndcg': ndcg,
                'count': count
            }
            
            print(f"{bucket.capitalize():<8}: HR@{K}={hr:.4f}, NDCG@{K}={ndcg:.4f}, N={count}")
        
        results[model_name] = model_results
    
    return results

# Define models for evaluation
models_to_evaluate = {
    "Text Model": text_model
}

# Load title embeddings if not already available for text model
try:
    title_embeddings_tensor
except NameError:
    title_embeddings = np.load('../data/embeddings/title_embeddings.npy')
    title_embeddings_tensor = torch.from_numpy(title_embeddings).float().to(device)
    print(f"Loaded title embeddings: {title_embeddings_tensor.shape}")

# Run bucketed evaluation
evaluation_results = bucketed_evaluation(models_to_evaluate, test_df, K=10)

# Print comparative summary
print(f"\n{'='*80}")
print("SUMMARY - HR@10 and NDCG@10")
print(f"{'='*80}")
model_name = "Text Model"
overall_hr = evaluation_results[model_name]['overall']['hr']
cold_hr = evaluation_results[model_name]['cold']['hr']
warm_hr = evaluation_results[model_name]['warm']['hr']  
hot_hr = evaluation_results[model_name]['hot']['hr']   
print(f"{model_name:<15} {overall_hr:<12.4f} {cold_hr:<12.4f} {warm_hr:<12.4f} {hot_hr:<12.4f}")



Item Popularity Distribution:
Cold items (≤5 ratings): 116962
Warm items (6-50 ratings): 9896
Hot items (>50 ratings): 385
Total items: 127243

Test users after filtering: 41847

Test item distribution:
Cold: 14974 test interactions
Warm: 12263 test interactions
Hot: 14610 test interactions

EVALUATING TEXT MODEL
Cold    : HR@10=0.0000, NDCG@10=0.0000, N=14974
Warm    : HR@10=0.0011, NDCG@10=0.0005, N=12263
Hot     : HR@10=0.0287, NDCG@10=0.0149, N=14610
Overall : HR@10=0.0103, NDCG@10=0.0053, N=41847

SUMMARY - HR@10 and NDCG@10
Text Model      0.0103       0.0000       0.0011       0.0287      
